# Silver layer
From bronze to silver following are followed:
- Created scd2 tables -(Patient, Payers, Providers, Policies), Performed intial load, implemented merge logic
- Created trasactional tables (Claims, CLiam_lines, payments)- performed load from bronze to silver
 


In [0]:
spark.sql("USE CATALOG healthcare_adjudication")

bronze_schema = "healthcare_adjudication_bronze"
silver_schema = "healthcare_adjudication_silver"

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_schema}.silver_patient (
    patient_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    patient_id STRING,
    patient_name STRING,
    gender STRING,
    dob DATE,
    state STRING,
    effective_start_date TIMESTAMP,
    effective_end_date TIMESTAMP,
    is_current STRING,
    version INT
)
USING DELTA
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_schema}.silver_provider (
    provider_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    provider_id STRING,
    provider_name STRING,
    specialty STRING,
    state STRING,
    effective_start_date TIMESTAMP,
    effective_end_date TIMESTAMP,
    is_current STRING,
    version INT
)
USING DELTA
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_schema}.silver_payer (
    payer_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    payer_id STRING,
    payer_name STRING,
    plan_type STRING,
    effective_start_date TIMESTAMP,
    effective_end_date TIMESTAMP,
    is_current STRING,
    version INT
)
USING DELTA
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_schema}.silver_policy (
    policy_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    policy_id STRING,
    payer_id STRING,
    plan STRING,
    coverage_limit DOUBLE,
    copay DOUBLE,
    effective_start_date TIMESTAMP,
    effective_end_date TIMESTAMP,
    is_current STRING,
    version INT
)
USING DELTA
""")

In [0]:
spark.sql(f"""
INSERT INTO {silver_schema}.silver_patient
(
patient_id,
patient_name,
gender,
dob,
state,
effective_start_date,
effective_end_date,
is_current
)
SELECT
patient_id,
name,
gender,
dob,
state,
current_timestamp(),
TIMESTAMP '9999-12-31',
'Y'
FROM {bronze_schema}.patients
""")

In [0]:
spark.sql(f"""
INSERT INTO {silver_schema}.silver_provider
(
provider_id,
provider_name,
specialty,
state,
effective_start_date,
effective_end_date,
is_current
)
SELECT
provider_id,
provider_name,
specialty,
state,
current_timestamp(),
TIMESTAMP '9999-12-31',
'Y'
FROM {bronze_schema}.providers
""")

In [0]:
spark.sql(f"""
INSERT INTO {silver_schema}.silver_payer
(
payer_id,
payer_name,
plan_type,
effective_start_date,
effective_end_date,
is_current
)
SELECT
payer_id,
payer_name,
plan_type,
current_timestamp(),
TIMESTAMP '9999-12-31',
'Y'
FROM {bronze_schema}.payers
""")

In [0]:
spark.sql(f"""
INSERT INTO {silver_schema}.silver_policy
(
policy_id,
payer_id,
plan,
coverage_limit,
copay,
effective_start_date,
effective_end_date,
is_current
)
SELECT
policy_id,
payer.payer_id,
coverage.plan,
coverage.limit,
coverage.copay,
current_timestamp(),
TIMESTAMP '2000-12-31',
'Y'
FROM {bronze_schema}.policies
""")

In [0]:
%sql
MERGE INTO healthcare_adjudication.healthcare_adjudication_silver.silver_patient AS target
USING (
    SELECT
        patient_id,
        name,
        gender,
        TO_DATE(dob,'yyyy-MM-dd') AS dob,
        state
    FROM healthcare_adjudication.healthcare_adjudication_bronze.patients
) AS source

ON target.patient_id = source.patient_id
AND target.is_current = 'Y'

WHEN MATCHED AND (
    target.patient_name <> source.name
    OR target.gender <> source.gender
    OR target.state <> source.state
)

THEN UPDATE SET
    target.is_current = 'N',
    target.effective_end_date = current_timestamp()

WHEN NOT MATCHED
THEN INSERT (
    patient_id,
    patient_name,
    gender,
    dob,
    state,
    effective_start_date,
    effective_end_date,
    is_current
)

VALUES (
    source.patient_id,
    source.name,
    source.gender,
    source.dob,
    source.state,
    current_timestamp(),
    TIMESTAMP '9999-12-31',
    'Y'
)
WHEN NOT MATCHED BY SOURCE THEN
  UPDATE SET
  target.is_current = 'N',
  target.effective_end_date = current_date()

In [0]:
%sql
MERGE INTO healthcare_adjudication.healthcare_adjudication_silver.silver_provider AS target

USING (
SELECT
provider_id,
provider_name,
specialty,
state
FROM healthcare_adjudication.healthcare_adjudication_bronze.providers
) AS source

ON target.provider_id = source.provider_id
AND target.is_current = 'Y'

WHEN MATCHED AND (
target.provider_name <> source.provider_name
OR target.specialty <> source.specialty
OR target.state <> source.state
)

THEN UPDATE SET
target.is_current = 'N',
target.effective_end_date = current_timestamp()

WHEN NOT MATCHED
THEN INSERT (
provider_id,
provider_name,
specialty,
state,
effective_start_date,
effective_end_date,
is_current
)

VALUES (
source.provider_id,
source.provider_name,
source.specialty,
source.state,
current_timestamp(),
TIMESTAMP '9999-12-31',
'Y'
)
WHEN NOT MATCHED BY SOURCE THEN
  UPDATE SET
  target.is_current = 'N',
  target.effective_end_date = current_date()

In [0]:
%sql
select * from healthcare_adjudication.healthcare_adjudication_silver.silver_provider where provider_id = '2001'

In [0]:
%sql
MERGE INTO healthcare_adjudication.healthcare_adjudication_silver.silver_payer AS target

USING (
SELECT
payer_id,
payer_name,
plan_type
FROM healthcare_adjudication.healthcare_adjudication_bronze.payers
) AS source

ON target.payer_id = source.payer_id
AND target.is_current = 'Y'

WHEN MATCHED AND (
target.payer_name <> source.payer_name
OR target.plan_type <> source.plan_type
)

THEN UPDATE SET
target.is_current = 'N',
target.effective_end_date = current_timestamp()

WHEN NOT MATCHED
THEN INSERT (
payer_id,
payer_name,
plan_type,
effective_start_date,
effective_end_date,
is_current
)

VALUES (
source.payer_id,
source.payer_name,
source.plan_type,
current_timestamp(),
TIMESTAMP '9999-12-31',
'Y'
)
WHEN NOT MATCHED BY SOURCE THEN
  UPDATE SET
target.is_current = 'N',
target.effective_end_date = current_date()

In [0]:
%sql
MERGE INTO healthcare_adjudication.healthcare_adjudication_silver.silver_policy AS target

USING (

SELECT *
FROM (

SELECT
policy_id,
payer.payer_id AS payer_id,
coverage.plan AS plan,
coverage.limit AS coverage_limit,
coverage.copay AS copay,
ROW_NUMBER() OVER (PARTITION BY policy_id ORDER BY policy_id) AS rn

FROM healthcare_adjudication.healthcare_adjudication_bronze.policies

)

WHERE rn = 1

) source

ON target.policy_id = source.policy_id
AND target.is_current = 'Y'

WHEN MATCHED AND (
target.payer_id <> source.payer_id
OR target.plan <> source.plan
OR target.coverage_limit <> source.coverage_limit
OR target.copay <> source.copay
)

THEN UPDATE SET
target.is_current = 'N',
target.effective_end_date = current_timestamp()

WHEN NOT MATCHED

THEN INSERT (
policy_id,
payer_id,
plan,
coverage_limit,
copay,
effective_start_date,
effective_end_date,
is_current
)

VALUES (
source.policy_id,
source.payer_id,
source.plan,
source.coverage_limit,
source.copay,
current_timestamp(),
TIMESTAMP '9999-12-31',
'Y'
)
WHEN NOT MATCHED BY SOURCE THEN
  UPDATE SET
  target.is_current = 'N',
  target.effective_end_date = current_date()

##Transactional tables

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_schema}.silver_claims (
    claim_id STRING,
    patient_id STRING,
    provider_id STRING,
    policy_id STRING,
    claim_date DATE,
    total_amount DOUBLE,
    status STRING,
    ingestion_timestamp TIMESTAMP
)
USING DELTA
""")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS healthcare_adjudication.healthcare_adjudication_silver.silver_claim_lines (
    claim_id STRING,
    procedure_code STRING,
    line_amount DOUBLE,
    ingestion_time TIMESTAMP
)
USING DELTA

In [0]:
%sql
CREATE TABLE IF NOT EXISTS healthcare_adjudication.healthcare_adjudication_silver.silver_payments (
    payment_id STRING,
    claim_id STRING,
    payer_id STRING,
    payment_amount DOUBLE,
    payment_status STRING,
    ingestion_time TIMESTAMP
)
USING DELTA

In [0]:
%sql
INSERT INTO healthcare_adjudication.healthcare_adjudication_silver.silver_claims

SELECT
claim_id,
patient_id,
provider_id,
policy_id,
TO_DATE(claim_date,'yyyy-MM-dd') AS claim_date,
CAST(claim_amount AS DOUBLE),
status,
ingestion_time

FROM healthcare_adjudication.healthcare_adjudication_bronze.claims

In [0]:
%sql
INSERT INTO healthcare_adjudication.healthcare_adjudication_silver.silver_claim_lines

SELECT
claim_id,
procedure.code AS procedure_code,
procedure.charge AS line_amount,
ingestion_time

FROM healthcare_adjudication.healthcare_adjudication_bronze.claim_lines
LATERAL VIEW explode(procedures) p AS procedure

In [0]:
%sql
INSERT INTO healthcare_adjudication.healthcare_adjudication_silver.silver_payments

SELECT
payment_id,
claim_id,
payer_id,
CAST(amount AS DOUBLE),
status,
ingestion_time

FROM healthcare_adjudication.healthcare_adjudication_bronze.payments